In [28]:
# ==============================================================================
# STEP 1: ENVIRONMENT SETUP & REPOSITORY INTEGRATION
# ==============================================================================
import os
import sys

# 1. Clone your Stress Inference repository from GitHub
# This ensures we are testing the actual modular code you've written.
!git clone https://github.com/AnGoraGou/CV-Stress-Inference-Pipeline.git

# 2. Add the repository to the Python path
# This allows us to import your custom modules (e.g., from src.pipeline)
# anywhere in this notebook.
sys.path.append('/content/CV-Stress-Inference-Pipeline')

# 3. Install required dependencies
# We use -q (quiet) to keep the notebook clean.
!pip install -q mediapipe opencv-python-headless pyVHR pandas numpy scipy matplotlib scikit-learn

# 4. Verify the directory structure exists based on the provided images
# The IMVIA-NIR dataset is expected to have 'still' and 'talking' folders.
from google.colab import drive
drive.mount('/content/drive')

# DATASET_PATH = "/content/IMVIA-NIR"
# SUBSETS = ["still", "talking"]

# print("Environment setup complete. Repository cloned.")

fatal: destination path 'CV-Stress-Inference-Pipeline' already exists and is not an empty directory.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
 DATASET_PATH = "/content/drive/MyDrive/IMVIA-NIR"
# You can now use this path to read/write files

# SUBSETS = ["still", "talking"]

# print("Environment setup complete. Repository cloned.")

In [31]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==============================================================================
# STEP 2: DATA ADAPTERS & METRICS
# ==============================================================================

def sync_and_resample_bvp(csv_path, target_frame_count):
    """
    Loads BVP data at 64Hz and resamples it to match the video frame count.
    Matches logic from the dataset's loadFiles.py.
    """
    # Load BVP data as a flat array
    # Based on loadFiles.py, we assume a single-column CSV of values
    try:
        bvp_df = pd.read_csv(csv_path, header=None)
        # Note: Some Empatica files have 2 header rows (timestamp/freq).
        # If yours does, use df.iloc[2:]
        bvp_raw = bvp_df.values.flatten()

        # Resample the 64Hz signal to match the video timeline (target_frame_count)
        # This is critical for frame-by-frame stress correlation.
        resampled_bvp = resample(bvp_raw, target_frame_count)

        # Normalize Ground Truth to [0, 100] for direct comparison with Stress_Index
        norm_bvp = (resampled_bvp - np.min(resampled_bvp)) / (np.max(resampled_bvp) - np.min(resampled_bvp) + 1e-6) * 100
        return norm_bvp
    except Exception as e:
        print(f"Error loading BVP {csv_path}: {e}")
        return None

def calculate_evaluation_metrics(ground_truth, predicted):
    """
    Computes key performance indicators for physiological tracking.
    """
    # Filter out any NaNs (common if face detection fails in 'talking' videos)
    mask = ~np.isnan(predicted)
    gt_clean = ground_truth[mask]
    pred_clean = predicted[mask]

    if len(pred_clean) == 0:
        return {"MAE": 0, "RMSE": 0, "Pearson_R": 0}

    # Mean Absolute Error: Average units of deviation (0-100 scale)
    mae = mean_absolute_error(gt_clean, pred_clean)

    # RMSE: Penalizes larger 'spikes' caused by motion artifacts
    rmse = np.sqrt(mean_squared_error(gt_clean, pred_clean))

    # Pearson R: Measures how well the predicted stress TRENDS with the BVP
    corr, _ = pearsonr(gt_clean, pred_clean) if len(gt_clean) > 1 else (0, 0)

    return {"MAE": mae, "RMSE": rmse, "Pearson_R": corr}

In [32]:
# ==============================================================================
# STEP 3: BATCH EVALUATION LOOP
# ==============================================================================

def run_test_on_dataset():
    all_subject_metrics = []

    for subset in SUBSETS:
        # Construct path like: /content/IMVIA-NIR/still/01/
        subset_dir = os.path.join(DATASET_PATH, subset)

        # Get subject folders 01-10
        folders = sorted([f for f in os.listdir(subset_dir) if os.path.isdir(os.path.join(subset_dir, f))])

        for subj in folders:
            subject_path = os.path.join(subset_dir, subj)
            video_file = os.path.join(subject_path, "vid.avi") # Based on loadFiles.py
            bvp_file = os.path.join(subject_path, "BVP.csv")

            if not os.path.exists(video_file): continue

            print(f"--> Processing {subset} Subject {subj}...")

            # --- 1. RUN YOUR REPOSITORY PIPELINE ---
            # Replace the lines below with your actual class/module call from the repo.
            # Example:
            # from src.inference import StressPipeline
            # pipe = StressPipeline()
            # predicted_scores = pipe.analyze_video(video_file)

            # For demonstration, we simulate the output length based on video metadata
            cap = cv2.VideoCapture(video_file)
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()

            # Placeholder: Replace with actual pipeline output
            mock_predicted_stress = np.random.uniform(40, 60, frame_count)

            # --- 2. SYNC GROUND TRUTH ---
            gt_bvp = sync_and_resample_bvp(bvp_file, frame_count)

            if gt_bvp is not None:
                # --- 3. EVALUATE ---
                metrics = calculate_evaluation_metrics(gt_bvp, mock_predicted_stress)
                metrics.update({"Type": subset, "Subject": subj})
                all_subject_metrics.append(metrics)

    return pd.DataFrame(all_subject_metrics)

# Run the full test
results_df = run_test_on_dataset()

# ==============================================================================
# STEP 4: FINAL EVALUATION SUMMARY
# ==============================================================================
print("\n" + "="*40)
print("      FINAL STRESS PIPELINE METRICS")
print("="*40)

if not results_df.empty:
    # We group by 'Type' to show if the pipeline is robust to motion (Talking vs Still)
    #
    summary = results_df.groupby("Type")[["MAE", "RMSE", "Pearson_R"]].mean()
    print(summary)

    # Display individual subject performance for deep dive
    print("\nDetailed Subject Breakdown:")
    print(results_df.sort_values(by=["Type", "Subject"]))
else:
    print("Error: No data was found at /content/IMVIA-NIR. Please verify dataset upload.")

--> Processing talking Subject 01...
--> Processing talking Subject 02...

      FINAL STRESS PIPELINE METRICS
               MAE       RMSE  Pearson_R
Type                                    
talking  12.088017  15.206187   0.021146

Detailed Subject Breakdown:
         MAE       RMSE  Pearson_R     Type Subject
0  11.917742  15.243338   0.042181  talking      01
1  12.258292  15.169036   0.000110  talking      02


In [ ]:
b